In [1]:
import json
import os
from pathlib import Path

with open('data/raw/finagent_config.json') as f:
    cfg = json.load(f)

TICKERS     = cfg['TICKERS']
MACRO       = cfg['MACRO']
ALL_SYMBOLS = cfg['ALL_SYMBOLS']
PERIOD      = cfg['PERIOD']
INTERVAL    = cfg['INTERVAL']
GROQ_API_KEY = cfg['GROQ_API_KEY']
NEWSAPI_KEY  = cfg['NEWSAPI_KEY']
FRED_API_KEY = cfg['FRED_API_KEY']

print(f'Tickers : {TICKERS}  |  Period : {PERIOD}')

Tickers : ['AAPL', 'MSFT', 'NVDA', 'GOOG']  |  Period : 5y


In [2]:
# !pip install groq

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import groq
from datetime import datetime, timedelta

In [4]:
raw_dfs = {}
for symbol in ALL_SYMBOLS:
    safe = symbol.replace('=', '_')
    path = f'data/raw/{safe}_prices.csv'
    if Path(path).exists():
        df = pd.read_csv(path, index_col='Date', parse_dates=True)
        raw_dfs[symbol] = df
        print(f'  ✅ Loaded {symbol:10} {len(df):4d} rows')
    else:
        print(f'  ⚠️  Missing: {path} — chạy Module 1 trước')
print(f'\n{len(raw_dfs)}/{len(ALL_SYMBOLS)} datasets loaded')

  ✅ Loaded AAPL         60 rows
  ✅ Loaded MSFT         60 rows
  ✅ Loaded NVDA         60 rows
  ✅ Loaded GOOG         60 rows
  ✅ Loaded GC=F         51 rows
  ✅ Loaded CL=F         51 rows

6/6 datasets loaded


# Clean & Feature Engineering

In [5]:
def clean_df(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().title() for c in df.columns]
    price_cols = [c for c in ['Open','High','Low','Close','Volume'] if c in df.columns]
    df = df[price_cols]
    for col in ['Open','High','Low','Close']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64') #Normalize price
    if 'Volume' in df.columns:
        df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce').fillna(0).astype('int64') #Normalize Volume
    df.sort_index(inplace=True)
    n_dup = df.index.duplicated().sum()
    if n_dup:
        df = df[~df.index.duplicated(keep='first')]
        print(f'  {symbol}: removed {n_dup} duplicates') #Duplicate records
    n_miss = df['Close'].isna().sum()
    if n_miss:
        print(f'  {symbol}: filling {n_miss} missing Close values')
    price_only = [c for c in ['Open','High','Low','Close'] if c in df.columns]
    df[price_only] = df[price_only].ffill(limit=2).interpolate(method='time', limit_direction='forward')
    df.dropna(subset=['Close'], inplace=True) # Missing values
    ret = df['Close'].pct_change()
    q1, q3 = ret.quantile(0.25), ret.quantile(0.75)
    iqr = q3 - q1
    df['Outlier_Flag'] = (ret < q1 - 3*iqr) | (ret > q3 + 3*iqr) | (ret.abs() > 0.40) #Outliers
    return df

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    c = df['Close']
    df['Daily_Return'] = c.pct_change()
    df['Log_Return']   = np.log(c / c.shift(1))
    df['MA_7']   = c.rolling(7).mean()
    df['MA_30']  = c.rolling(30).mean()
    df['EMA_12'] = c.ewm(span=12, adjust=False).mean()
    df['EMA_26'] = c.ewm(span=26, adjust=False).mean()
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    roll_mean = c.rolling(20).mean()
    roll_std  = c.rolling(20).std()
    df['BB_Middle'] = roll_mean
    df['BB_Upper']  = roll_mean + 2 * roll_std
    df['BB_Lower']  = roll_mean - 2 * roll_std
    df['Volatility_30'] = df['Log_Return'].rolling(30).std() * np.sqrt(12)
    delta = c.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    df['RSI_14'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))
    return df

print('Cleaning & engineering …')
print('-' * 40)
clean_dfs = {}
for symbol in ALL_SYMBOLS:
    if symbol not in raw_dfs:
        continue
    df = clean_df(raw_dfs[symbol], symbol)
    df = engineer_features(df)
    safe = symbol.replace('=', '_')
    df.to_csv(f'data/processed/{safe}_clean.csv')
    clean_dfs[symbol] = df
    print(f'  ✅ {symbol:10} {len(df):4d} rows, {len(df.columns):2d} cols → data/processed/{safe}_clean.csv')
print(f'\n✅ Cleaning complete: {len(clean_dfs)} datasets')

Cleaning & engineering …
----------------------------------------
  ✅ AAPL         60 rows, 19 cols → data/processed/AAPL_clean.csv
  ✅ MSFT         60 rows, 19 cols → data/processed/MSFT_clean.csv
  ✅ NVDA         60 rows, 19 cols → data/processed/NVDA_clean.csv
  ✅ GOOG         60 rows, 19 cols → data/processed/GOOG_clean.csv
  ✅ GC=F         51 rows, 19 cols → data/processed/GC_F_clean.csv
  ✅ CL=F         51 rows, 19 cols → data/processed/CL_F_clean.csv

✅ Cleaning complete: 6 datasets


# Fundamental data

In [6]:
FUNDAMENTAL_FIELDS = {
    'shortName':        'Company',
    'marketCap':        'Market Cap ($)',
    'trailingPE':       'P/E Ratio',
    'forwardPE':        'Forward P/E',
    'trailingEps':      'EPS (TTM)',
    'revenuePerShare':  'Revenue/Share',
    'totalRevenue':     'Total Revenue ($)',
    'grossMargins':     'Gross Margin',
    'profitMargins':    'Profit Margin',
    'debtToEquity':     'Debt/Equity',
    'returnOnEquity':   'ROE',
    'dividendYield':    'Dividend Yield',
    'fiftyTwoWeekHigh': '52W High ($)',
    'fiftyTwoWeekLow':  '52W Low ($)',
    'beta':             'Beta',
}
PCT_FIELDS  = {'grossMargins', 'profitMargins', 'dividendYield', 'returnOnEquity'}
SIZE_FIELDS = {'marketCap', 'totalRevenue'}

TICKER_FALLBACK = {
    'GOOG': 'GOOGL',
}

def format_value(yf_key, val):
    """Format a single fundamental value for display."""
    if val is None:
        return 'N/A'
    if yf_key in PCT_FIELDS:
        return f'{val * 100:.2f}%'
    if yf_key in SIZE_FIELDS:
        if val >= 1e12: return f'${val / 1e12:.2f}T'
        if val >= 1e9:  return f'${val / 1e9:.2f}B'
        return f'${val / 1e6:.2f}M'
    if isinstance(val, float):
        return round(val, 2)
    return val

def fetch_fundamentals(tickers):
    records = {}
    for ticker in tickers:
        try:
            info = yf.Ticker(ticker).info
            records[ticker] = {
                label: format_value(yf_key, info.get(yf_key))
                for yf_key, label in FUNDAMENTAL_FIELDS.items()
            }
            print(f'  ✅ {ticker} — P/E: {info.get("trailingPE","N/A")}  EPS: {info.get("trailingEps","N/A")}')
        except Exception as e:
            print(f'  ❌ {ticker}: {e}')
    return pd.DataFrame(records).T

print('Fundamental data …')
fund_df = fetch_fundamentals(TICKERS)
fund_df.to_csv('data/processed/fundamentals.csv')
print('\n📊 Fundamentals:')
display(fund_df.T)

Fundamental data …
  ✅ AAPL — P/E: 37.432728  EPS: 8.25
  ✅ MSFT — P/E: 24.944576  EPS: 16.78
  ✅ NVDA — P/E: 33.026073  EPS: 6.52
  ✅ GOOG — P/E: 28.894136  EPS: 13.13

📊 Fundamentals:


,AAPL,MSFT,NVDA,GOOG
Company,Apple Inc.,Microsoft Corporation,NVIDIA Corporation,Alphabet Inc.
Market Cap ($),$4.54T,$3.11T,$5.22T,$4.60T
P/E Ratio,37.43,24.94,33.03,28.89
Forward P/E,32.16,21.65,17.03,26.25
EPS (TTM),8.25,16.78,6.52,13.13
Revenue/Share,30.53,42.84,10.42,34.93
Total Revenue ($),$451.44B,$318.27B,$253.49B,$422.50B
Gross Margin,47.86%,68.31%,74.14%,60.37%
Profit Margin,27.15%,39.34%,62.97%,37.92%
Debt/Equity,79.55,30.27,6.55,20.03


# Macro Data

In [7]:
FRED_BASE = 'https://api.stlouisfed.org/fred/series/observations'
end_date  = datetime.today().strftime('%Y-%m-%d')
_period_years = int(PERIOD.replace('y', ''))
start_date = (datetime.today() - timedelta(days=365*_period_years)).strftime('%Y-%m-%d')
FRED_SERIES = {
    'FEDFUNDS':'Fed Funds Rate (%)','CPIAUCSL':'CPI (Index)',
    'UNRATE':'Unemployment Rate (%)','DGS10':'10Y Treasury Yield (%)',
}

def fetch_fred(series_id, label, api_key):
    params = {'series_id':series_id,'api_key':api_key,'file_type':'json',
              'observation_start':start_date,'observation_end':end_date}
    resp = requests.get(FRED_BASE, params=params, timeout=10)
    resp.raise_for_status()
    obs = resp.json().get('observations',[])
    s = pd.Series({o['date']:float(o['value']) for o in obs if o['value']!='.'}, name=label)
    s.index = pd.to_datetime(s.index)
    return s

def make_demo_macro():
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    np.random.seed(42)
    df = pd.DataFrame(index=dates)
    df['Fed Funds Rate (%)']     = np.linspace(4.5, 5.25, len(dates)) + np.random.normal(0,.05,len(dates))
    df['CPI YoY Inflation (%)']  = np.linspace(6.0, 3.2, len(dates)) + np.random.normal(0,.15,len(dates))
    df['Unemployment Rate (%)']  = np.linspace(3.6, 3.9, len(dates)) + np.random.normal(0,.08,len(dates))
    df['10Y Treasury Yield (%)'] = np.linspace(3.8, 4.5, len(dates)) + np.random.normal(0,.12,len(dates))
    print('  ⚠️  DEMO macro data — set FRED_API_KEY for real data')
    return df.round(2)

print('Macro data (FRED) …')
if FRED_API_KEY and FRED_API_KEY != 'your_fred_key_here':
    try:
        series_list = []
        for sid, label in FRED_SERIES.items():
            s = fetch_fred(sid, label, FRED_API_KEY)
            if sid == 'CPIAUCSL':
                s = s.pct_change(12)*100; s.name = 'CPI YoY Inflation (%)'
            series_list.append(s)
            print(f'  ✅ {sid} — {len(s)} obs')
        macro_df = pd.concat(series_list, axis=1).resample('MS').last().dropna(how='all')
    except Exception as e:
        print(f'  ❌ {e} — using demo data')
        macro_df = make_demo_macro()
else:
    macro_df = make_demo_macro()
macro_df.index.name = 'Date'
macro_df.to_csv('data/processed/macro_fred.csv')
print(f'\n✅ Macro data: {len(macro_df)} months saved')

Macro data (FRED) …
  ✅ FEDFUNDS — 60 obs
  ✅ CPIAUCSL — 59 obs
  ✅ UNRATE — 59 obs
  ✅ DGS10 — 1249 obs

✅ Macro data: 61 months saved


C:\Users\VIP\AppData\Local\Temp\ipykernel_13356\954797747.py:41: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  macro_df = pd.concat(series_list, axis=1).resample('MS').last().dropna(how='all')
